# 🧠 Beyin MR Görüntülerinden Tümör Sınıflandırması
## Derin Öğrenme ve Grad-CAM ile Açıklanabilirlik Analizi

Bu notebook, beyin MR görüntülerini 4 sınıfa ayıran iki farklı modeli (Custom CNN ve EfficientNetB0) eğitir, karşılaştırır ve Grad-CAM ile en başarılı modelin kararlarını görselleştirir.

**Sınıflar:** glioma, meningioma, notumor, pituitary
**Veri seti:** Brain Tumor MRI Dataset (Kaggle - Masoud Nickparvar)
**Giriş boyutu:** 224×224×3

---


## 1. Kurulum ve Kütüphaneler

In [ ]:
# Google Colab GPU kontrolü
import tensorflow as tf
print("TensorFlow versiyonu:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("✓ GPU hazır")
else:
    print("⚠ UYARI: GPU bulunamadı! Runtime > Change runtime type > GPU seçin.")

In [ ]:
# Reproducibility için sabit seed
import numpy as np
import random
import os
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
print(f"✓ Random seed sabitlendi: {SEED}")

In [ ]:
# Gerekli kütüphaneler
import os
import shutil
import zipfile
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_recall_fscore_support,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# Görselleştirme ayarları
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
print("✓ Tüm kütüphaneler yüklendi")

## 2. Veri Seti İndirme

İki yöntem var:

**A) Kaggle API ile (önerilen):** Kaggle hesabınızdan `kaggle.json` API anahtarınızı indirip Colab'a yükleyin.

**B) Manuel:** [Bu linkten](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) zip olarak indirip Colab'a yükleyin.


In [ ]:
# A YÖNTEMİ: Kaggle API ile (kaggle.json'ı Colab'a yükledikten sonra)
# Eğer kaggle.json zaten yüklediyseniz bu hücreyi çalıştırın
try:
    from google.colab import files
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print("kaggle.json dosyasını seçin:")
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    
    !pip install -q kaggle
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
    
    # Zip'ten çıkar
    if os.path.exists('brain-tumor-mri-dataset.zip'):
        with zipfile.ZipFile('brain-tumor-mri-dataset.zip', 'r') as z:
            z.extractall('data')
        print("✓ Veri seti çıkarıldı: data/")
except Exception as e:
    print(f"Kaggle API ile indirme yapılamadı: {e}")
    print("Lütfen B yöntemini deneyin (manuel yükleme).")

In [ ]:
# Veri seti yapısını kontrol et
TRAIN_DIR = 'data/Training'
TEST_DIR = 'data/Testing'

if os.path.exists(TRAIN_DIR):
    print("Eğitim klasörü içeriği:")
    for cls in sorted(os.listdir(TRAIN_DIR)):
        n = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
        print(f"  {cls}: {n} görüntü")
    print("\nTest klasörü içeriği:")
    for cls in sorted(os.listdir(TEST_DIR)):
        n = len(os.listdir(os.path.join(TEST_DIR, cls)))
        print(f"  {cls}: {n} görüntü")
else:
    print("⚠ data/Training klasörü bulunamadı. Veri setini yükleyin.")

## 3. Veri Keşfi ve Görselleştirme

In [ ]:
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4

# Sınıf dağılımı görselleştirmesi
train_counts = {cls: len(os.listdir(os.path.join(TRAIN_DIR, cls))) for cls in CLASS_NAMES}
test_counts = {cls: len(os.listdir(os.path.join(TEST_DIR, cls))) for cls in CLASS_NAMES}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(train_counts.keys(), train_counts.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0].set_title('Eğitim Seti Sınıf Dağılımı', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Görüntü Sayısı')
for i, (k, v) in enumerate(train_counts.items()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].bar(test_counts.keys(), test_counts.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1].set_title('Test Seti Sınıf Dağılımı', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Görüntü Sayısı')
for i, (k, v) in enumerate(test_counts.items()):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Toplam eğitim: {sum(train_counts.values())}")
print(f"Toplam test: {sum(test_counts.values())}")

In [ ]:
# Her sınıftan örnek görüntüler
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    samples = os.listdir(cls_dir)[:4]
    for j, sample in enumerate(samples):
        img = cv2.imread(os.path.join(cls_dir, sample))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(img)
        axes[i, j].set_title(f'{cls}', fontsize=11)
        axes[i, j].axis('off')
plt.suptitle('Her Sınıftan Örnek MR Görüntüleri', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Veri Hazırlama ve Augmentation

Eğitim verisine **veri artırma** uyguluyoruz:
- Rotation ±15°
- Width/Height shift ±10%
- Zoom ±10%
- Horizontal flip
- Brightness ±10%

Doğrulama ve test için sadece normalizasyon yapıyoruz.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    validation_split=0.15,
    fill_mode='nearest'
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training',
    shuffle=True, seed=SEED, classes=CLASS_NAMES
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation',
    shuffle=False, seed=SEED, classes=CLASS_NAMES
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False, classes=CLASS_NAMES
)

print(f"\nEğitim örnekleri: {train_gen.samples}")
print(f"Doğrulama örnekleri: {val_gen.samples}")
print(f"Test örnekleri: {test_gen.samples}")

In [ ]:
# Sınıf ağırlıkları (dengesizliği telafi için)
classes = train_gen.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)
class_weight_dict = dict(enumerate(class_weights))
print("Sınıf ağırlıkları:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls}: {class_weight_dict[i]:.3f}")

## 5. Model 1: Custom CNN (Baseline)

Sıfırdan eğitilen 4-blok'lu konvolüsyonel sinir ağı.

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(256, (3, 3), padding='same', name='last_conv'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='CustomCNN')
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

In [ ]:
# Custom CNN eğitimi
EPOCHS = 20

cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_cnn.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

cnn_history = cnn_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, callbacks=cnn_callbacks,
    class_weight=class_weight_dict, verbose=1
)

## 6. Model 2: EfficientNetB0 (Transfer Learning)

ImageNet üzerinde önceden eğitilmiş EfficientNetB0 modeli + custom classification head. Son 30 katman fine-tuning için açıldı.

In [ ]:
def build_efficientnet(input_shape=(224, 224, 3), num_classes=4):
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = False
    # Son 30 katmanı fine-tuning için aç (BatchNorm hariç)
    for layer in base_model.layers[-30:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    
    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name='EfficientNetB0_TL')
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

eff_model = build_efficientnet()
eff_model.summary()

In [ ]:
# EfficientNetB0 eğitimi
eff_callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_efficientnet.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

eff_history = eff_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, callbacks=eff_callbacks,
    class_weight=class_weight_dict, verbose=1
)

## 7. Eğitim Eğrileri

In [ ]:
def plot_training_curves(history, model_name, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'], label='Eğitim', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Doğrulama', linewidth=2)
    axes[0].set_title(f'{model_name} - Doğruluk Eğrisi', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history.history['loss'], label='Eğitim', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Doğrulama', linewidth=2)
    axes[1].set_title(f'{model_name} - Loss Eğrisi', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_training_curves(cnn_history, 'Custom CNN', 'training_curves_cnn.png')
plot_training_curves(eff_history, 'EfficientNetB0', 'training_curves_efficientnet.png')

## 8. Model Değerlendirmesi

In [ ]:
def evaluate_model(model, test_gen, class_names, model_name):
    test_gen.reset()
    y_pred_proba = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = test_gen.classes
    
    acc = accuracy_score(y_true, y_pred)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro Precision: {mp:.4f} | Macro Recall: {mr:.4f} | Macro F1: {mf:.4f}")
    print(f"Weighted F1: {wf:.4f}")
    print("\nPer-class metrics:")
    for i, cls in enumerate(class_names):
        print(f"  {cls}: P={p[i]:.3f} R={r[i]:.3f} F1={f[i]:.3f} (n={s[i]})")
    
    return {
        'accuracy': acc, 'macro_precision': mp, 'macro_recall': mr,
        'macro_f1': mf, 'weighted_f1': wf,
        'precision_per_class': p, 'recall_per_class': r,
        'f1_per_class': f, 'support_per_class': s,
        'cm': confusion_matrix(y_true, y_pred),
        'y_true': y_true, 'y_pred': y_pred, 'y_pred_proba': y_pred_proba
    }

cnn_results = evaluate_model(cnn_model, test_gen, CLASS_NAMES, 'Custom CNN')
eff_results = evaluate_model(eff_model, test_gen, CLASS_NAMES, 'EfficientNetB0')

In [ ]:
# Confusion matrix'leri çiz
def plot_cm(cm, class_names, title, save_path):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Sayı'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('Gerçek Sınıf'); plt.xlabel('Tahmin Edilen Sınıf')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_cm(cnn_results['cm'], CLASS_NAMES, 'Custom CNN - Confusion Matrix', 'cm_cnn.png')
plot_cm(eff_results['cm'], CLASS_NAMES, 'EfficientNetB0 - Confusion Matrix', 'cm_efficientnet.png')

In [ ]:
# ROC eğrileri (One-vs-Rest)
def plot_roc(y_true, y_pred_proba, class_names, title, save_path):
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    plt.figure(figsize=(9, 7))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    aucs = []
    for i, (cls, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls} (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(loc='lower right'); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return aucs

cnn_aucs = plot_roc(cnn_results['y_true'], cnn_results['y_pred_proba'],
                     CLASS_NAMES, 'Custom CNN - ROC Eğrileri', 'roc_cnn.png')
eff_aucs = plot_roc(eff_results['y_true'], eff_results['y_pred_proba'],
                     CLASS_NAMES, 'EfficientNetB0 - ROC Eğrileri', 'roc_efficientnet.png')
print(f"\nOrtalama AUC - Custom CNN: {np.mean(cnn_aucs):.4f}")
print(f"Ortalama AUC - EfficientNetB0: {np.mean(eff_aucs):.4f}")

## 9. Model Karşılaştırma Tablosu

In [ ]:
# Karşılaştırma tablosu
comparison = pd.DataFrame({
    'Model': ['Custom CNN', 'EfficientNetB0'],
    'Accuracy': [f"{cnn_results['accuracy']:.4f}", f"{eff_results['accuracy']:.4f}"],
    'Macro Precision': [f"{cnn_results['macro_precision']:.4f}", f"{eff_results['macro_precision']:.4f}"],
    'Macro Recall': [f"{cnn_results['macro_recall']:.4f}", f"{eff_results['macro_recall']:.4f}"],
    'Macro F1': [f"{cnn_results['macro_f1']:.4f}", f"{eff_results['macro_f1']:.4f}"],
    'Weighted F1': [f"{cnn_results['weighted_f1']:.4f}", f"{eff_results['weighted_f1']:.4f}"],
    'Mean AUC': [f"{np.mean(cnn_aucs):.4f}", f"{np.mean(eff_aucs):.4f}"],
    'Parameters': [f"{cnn_model.count_params():,}", f"{eff_model.count_params():,}"]
})
comparison.to_csv('model_comparison.csv', index=False)
print(comparison.to_string(index=False))

In [ ]:
# Per-class detaylı metrik tablosu
def per_class_df(results, model_name, class_names):
    return pd.DataFrame({
        'Model': [model_name] * len(class_names),
        'Sınıf': class_names,
        'Precision': [f"{p:.4f}" for p in results['precision_per_class']],
        'Recall': [f"{r:.4f}" for r in results['recall_per_class']],
        'F1-Score': [f"{f:.4f}" for f in results['f1_per_class']],
        'Support': results['support_per_class']
    })

per_class_cnn = per_class_df(cnn_results, 'Custom CNN', CLASS_NAMES)
per_class_eff = per_class_df(eff_results, 'EfficientNetB0', CLASS_NAMES)
per_class_combined = pd.concat([per_class_cnn, per_class_eff], ignore_index=True)
per_class_combined.to_csv('per_class_metrics.csv', index=False)
print(per_class_combined.to_string(index=False))

## 10. Grad-CAM Açıklanabilirlik Analizi

En başarılı model üzerinde Grad-CAM uygulayarak modelin karar verirken görüntünün hangi bölgelerine odaklandığını görselleştiriyoruz.

In [ ]:
# En iyi modeli seç
if eff_results['accuracy'] >= cnn_results['accuracy']:
    best_model = eff_model
    best_model_name = 'EfficientNetB0'
    # EfficientNet için son conv katmanı
    last_conv_layer_name = 'top_conv'
else:
    best_model = cnn_model
    best_model_name = 'Custom CNN'
    last_conv_layer_name = 'last_conv'

print(f"✓ En iyi model: {best_model_name}")
print(f"✓ Son conv katmanı: {last_conv_layer_name}")

In [ ]:
# Grad-CAM fonksiyonları
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # EfficientNet için iç içe model yapısını çöz
    if isinstance(model.layers[1], tf.keras.Model):
        base = model.layers[1]
        conv_layer = base.get_layer(last_conv_layer_name)
        grad_model = tf.keras.models.Model(
            inputs=model.inputs,
            outputs=[conv_layer.output, model.output]
        )
    else:
        grad_model = tf.keras.models.Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv_layer_name).output, model.output]
        )
    
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

def overlay_heatmap(img, heatmap, alpha=0.4):
    if img.max() <= 1.0:
        img = (img * 255).astype(np.uint8)
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlayed = heatmap_color * alpha + img * (1 - alpha)
    return np.clip(overlayed, 0, 255).astype(np.uint8)
print("✓ Grad-CAM fonksiyonları hazır")

In [ ]:
# Test setinden her sınıftan 2 örnek seçip Grad-CAM uygula
test_gen.reset()
all_images, all_labels = [], []
for i in range(min(20, len(test_gen))):
    batch_x, batch_y = next(test_gen)
    all_images.append(batch_x)
    all_labels.append(batch_y)
all_images = np.concatenate(all_images)
all_labels = np.concatenate(all_labels)
all_labels_idx = np.argmax(all_labels, axis=1)

# Her sınıftan 2 örnek
selected_indices = []
for cls_idx in range(4):
    cls_samples = np.where(all_labels_idx == cls_idx)[0][:2]
    selected_indices.extend(cls_samples)

print(f"Seçilen örnek sayısı: {len(selected_indices)}")

In [ ]:
# Grad-CAM görsellerini grid olarak çiz
n_samples = len(selected_indices)
fig, axes = plt.subplots(n_samples, 3, figsize=(13, 4 * n_samples))

for row, idx in enumerate(selected_indices):
    img = all_images[idx]
    true_idx = all_labels_idx[idx]
    img_batch = np.expand_dims(img, axis=0)
    
    preds = best_model.predict(img_batch, verbose=0)
    pred_idx = np.argmax(preds[0])
    confidence = preds[0][pred_idx]
    
    heatmap = make_gradcam_heatmap(img_batch, best_model, last_conv_layer_name)
    overlay = overlay_heatmap(img, heatmap, alpha=0.4)
    
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'Orijinal MR\nGerçek: {CLASS_NAMES[true_idx]}', fontsize=11)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(heatmap, cmap='jet')
    axes[row, 1].set_title('Grad-CAM Isı Haritası', fontsize=11)
    axes[row, 1].axis('off')
    
    correct = '✓' if pred_idx == true_idx else '✗'
    color = 'green' if pred_idx == true_idx else 'red'
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(
        f'Bindirilmiş {correct}\nTahmin: {CLASS_NAMES[pred_idx]} ({confidence:.2%})',
        fontsize=11, color=color
    )
    axes[row, 2].axis('off')

plt.suptitle(f'Grad-CAM Analizi - {best_model_name}', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Örnek Tahminler (Gerçek vs Tahmin + Confidence)

In [ ]:
# 12 rastgele örnek üzerinde tahmin
np.random.seed(SEED)
random_idx = np.random.choice(len(all_images), 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, idx in enumerate(random_idx):
    img = all_images[idx]
    true_idx = all_labels_idx[idx]
    preds = best_model.predict(np.expand_dims(img, 0), verbose=0)
    pred_idx = np.argmax(preds[0])
    confidence = preds[0][pred_idx]
    
    correct = pred_idx == true_idx
    color = 'green' if correct else 'red'
    axes[i].imshow(img)
    axes[i].set_title(
        f'Gerçek: {CLASS_NAMES[true_idx]}\nTahmin: {CLASS_NAMES[pred_idx]} ({confidence:.2%})',
        fontsize=10, color=color
    )
    axes[i].axis('off')

plt.suptitle(f'{best_model_name} - Örnek Tahminler', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Sonuçları Drive'a Kaydetme

In [ ]:
# Tüm sonuç dosyalarını zip'le
import zipfile

result_files = [
    'class_distribution.png', 'sample_images.png',
    'training_curves_cnn.png', 'training_curves_efficientnet.png',
    'cm_cnn.png', 'cm_efficientnet.png',
    'roc_cnn.png', 'roc_efficientnet.png',
    'model_comparison.csv', 'per_class_metrics.csv',
    'gradcam_results.png', 'sample_predictions.png',
    'best_cnn.h5', 'best_efficientnet.h5'
]

with zipfile.ZipFile('project_results.zip', 'w') as zipf:
    for f in result_files:
        if os.path.exists(f):
            zipf.write(f)
            print(f"  ✓ {f}")

print("\n✓ Tüm sonuçlar 'project_results.zip' dosyasına paketlendi.")
print("Sol panelden indirip GitHub repo'nuzun results/ klasörüne kopyalayın.")

In [ ]:
# Eğitim geçmişlerini de kaydet (rapor için)
import json

history_data = {
    'cnn': {
        'accuracy': cnn_history.history['accuracy'],
        'val_accuracy': cnn_history.history['val_accuracy'],
        'loss': cnn_history.history['loss'],
        'val_loss': cnn_history.history['val_loss']
    },
    'efficientnet': {
        'accuracy': eff_history.history['accuracy'],
        'val_accuracy': eff_history.history['val_accuracy'],
        'loss': eff_history.history['loss'],
        'val_loss': eff_history.history['val_loss']
    },
    'final_results': {
        'cnn_accuracy': float(cnn_results['accuracy']),
        'cnn_macro_f1': float(cnn_results['macro_f1']),
        'eff_accuracy': float(eff_results['accuracy']),
        'eff_macro_f1': float(eff_results['macro_f1']),
        'best_model': best_model_name
    }
}

with open('training_history.json', 'w') as f:
    json.dump(history_data, f, indent=2)
print("✓ training_history.json kaydedildi")
print(f"\n🎯 En iyi model: {best_model_name}")
print(f"🎯 En iyi doğruluk: {max(cnn_results['accuracy'], eff_results['accuracy']):.4f}")

## ✅ Proje Tamamlandı!

Eğitim, değerlendirme ve Grad-CAM analizi başarıyla tamamlandı. `project_results.zip` dosyasını indirip GitHub repo'nuzun ilgili klasörlerine kopyalayın.

Sonraki adımlar:
1. ✅ Sonuçları GitHub'a yükle
2. ✅ README'yi güncelle (gerçek sonuç değerleri ile)
3. ✅ Raporu yaz
4. ✅ Broşürü hazırla
5. ✅ Demo video çek
